# Info-Driven Bar Trading — Run on Colab

> ⚠️ **This repository and notebook were created entirely by AI.**

**Set the runtime to T4 GPU** (`Runtime → Change runtime type → T4 GPU`). Assumes your gold `datasets/` folder is in Google Drive.

## 1. Get the code
**Fresh session — clone:**

In [ ]:
!git clone https://github.com/fedallah-jr/Info-Driven-Bar-Trading.git
%cd Info-Driven-Bar-Trading

**Already cloned — pull the latest code:**

In [ ]:
%cd /content/Info-Driven-Bar-Trading
!git pull

## 2. Mount Drive + stage the gold datasets into `data/datasets/`
Edit `DRIVE_DATASETS`. For the basket/pooled runs you need all 10 coins' `*_cusum_thr0.020.parquet`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DATASETS = '/content/drive/MyDrive/datasets'   # <-- your datasets/ folder
!mkdir -p data/datasets
!cp -v {DRIVE_DATASETS}/*.parquet data/datasets/
!echo 'datasets available:'; ls data/datasets/ | wc -l

## 3. GPU check

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## A. Reproduce the paper (v1 grid)
ETH/BTC × {CUSUM, range} × 2%, the paper's 2022-Q2…2023-Q2 window. ~20-40 min on T4.

In [ ]:
!python3 model/train.py --grid

## B. Robustness — extend out-of-sample to 2026 (the key test)
Same ETH-CUSUM cell, but tested across **2022→2026** (16 quarters of held-out data we never trained on). ~3x the trades — this is what tells us if the edge is real. **~1–1.5 h on T4.**

In [ ]:
!python3 model/train.py --coin ETHUSDT --bartype cusum --threshold 0.02 --barrier 0.05 --quarters extended --seeds 0,1,2

## C. More coins — full 10-coin basket, per-coin, extended OOS
One model per coin. 1 seed to keep it feasible (**~3–5 h**). Barrier 4% is a uniform default (not tuned per coin → exploratory).

In [ ]:
!python3 model/train.py --basket --bartype cusum --threshold 0.02 --barrier 0.04 --quarters extended --seeds 0

## D. Predict coins *together* — pooled cross-sectional model
**One** model trained on all 10 coins pooled (each scaled by its own train stats), predicting each coin's test set. The heaviest run — start with `--quarters paper` (5 quarters, ~30-60 min) to sanity-check before the extended version.

In [ ]:
# quick first: !python3 model/train.py --basket --pooled --bartype cusum --threshold 0.02 --barrier 0.04 --quarters paper --seeds 0
!python3 model/train.py --basket --pooled --bartype cusum --threshold 0.02 --barrier 0.04 --quarters extended --seeds 0

## E. A different architecture — Transformer (ETH, extended)
Different inductive bias from the conv/LSTM → trades differently. **~1–1.5 h.**

In [ ]:
!python3 model/train.py --coin ETHUSDT --model transformer --bartype cusum --threshold 0.02 --barrier 0.05 --quarters extended --seeds 0,1,2

## Evaluate everything → metrics table
Produces one row per `*_preds.parquet` in `results/model/` (per-coin, pooled, transformer, …).

In [ ]:
!python3 model/evaluate.py
from IPython.display import Markdown
Markdown(open('results/model/RESULTS.md').read())

### Notes
- Run **B first** — highest value per GPU-hour. C/D/E as time allows.
- Colab is ephemeral — persist outputs: `!cp -r results /content/drive/MyDrive/info-driven-results` (or pass `--out-dir /content/drive/MyDrive/info-driven-results` to `train.py`).
- Pooled/basket need all 10 coins' `*_cusum_thr0.020.parquet` in `data/datasets/`.